# Mock interview — full round-trip

This notebook covers the complete mock-interview lifecycle: start, generate
questions, ask, evaluate each answer, then move to the next question or compute
an overall score.

**Source files**
- `app.py` — `interview_start`, `_ask_question`, `interview_submit`
- `careercrew/interview.py` — `generate_questions`, `evaluate_answer`, `append_turn`, `overall_evaluation`
- Detail doc: `docs/CODE_PATH_MOCK_INTERVIEW.md`


## Overview

The mock interview requires Analyze context first (`ctx_state`).

On **Start**, CareerCrew generates grounded questions from the resume, JD, gaps,
and prep plan, then asks question zero with TTS and a transcript line.

On each **Submit**: STT or typed text → guardrail → evaluate → append the
candidate turn → either ask the next question the same way, or compute an
overall score when finished.

This is practice only — nothing is submitted to an employer.


## Flowchart

```
[Analyze done] ctx_state
        │
        ▼
 interview_start
   generate_questions(profile, requirements, gaps, prep_plan, n, llm)
   MockInterviewSession(...)
        │
        ▼
 _ask_question(0)  ──► TTS + INTERVIEWER turn + Chatbot
        │
        ▼
 interview_submit  ◄── mic filepath and/or typed text
   transcribe → answer text
   validate_interview_answer
   evaluate_answer → AnswerEvaluation
   append_turn(CANDIDATE)
        │
        ├── more questions? → _ask_question(next)
        └── last? → overall_evaluation → completed=True
```


## Step A — `interview_start`

1. `_load_context(ctx_state)` — must exist, or show “Analyze first”.
2. `interview.generate_questions(...)` — LLM JSON list, or `_fallback_questions` from gaps/prep.
3. Build `MockInterviewSession` with a new `session_id`.
4. Return `_ask_question(session, 0)`.


In [ ]:
# Offline model of question generation + session start

from dataclasses import dataclass, field

@dataclass
class Question:
    index: int
    text: str
    intent: str = ""

@dataclass
class Session:
    session_id: str
    questions: list
    turns: list = field(default_factory=list)
    evaluations: list = field(default_factory=list)
    completed: bool = False
    overall_score: int = 0

def fallback_questions(gaps, n=3):
    """Mirrors interview._fallback_questions: questions grounded in gaps."""
    base = [f"How would you approach: {g}?" for g in gaps] or [
        "Walk me through a recent project you owned."
    ]
    while len(base) < n:
        base.append(f"Follow-up competency question #{len(base)+1}")
    return [Question(i, t, intent="grounded practice") for i, t in enumerate(base[:n])]

gaps = ["system design for APIs", "ownership of production incidents", "Python typing"]
session = Session("iv-demo-001", fallback_questions(gaps, n=3))
print("session", session.session_id)
for q in session.questions:
    print(f"  Q{q.index}: {q.text}")


## Step B — `_ask_question`

| Action | Code |
|--------|------|
| TTS | `audio.speak(question.text)` |
| Transcript | `append_turn(INTERVIEWER, question.text)` |
| Persist | `_STORE.save_interview(session)` |
| UI output | question HTML, audio path, chatbot messages, `iv_state` JSON |


In [ ]:
def ask_question(session: Session, index: int):
    q = session.questions[index]
    print(f"[TTS] {q.text}")
    session.turns.append({"role": "interviewer", "q": index, "text": q.text})
    return q

ask_question(session, 0)
print("turns so far:", session.turns)


## Step C — `interview_submit`

Prefer STT over typed text → validate → `evaluate_answer` → append candidate turn.

**Evaluation fields:** `score`, `what_interviewer_wanted`, `strengths`,
`improvements`, `model_answer_outline`.

Then:
- if more questions remain → `_ask_question(next)` and show feedback HTML
- if this was the last answer → `overall_evaluation` (average scores + summary)


In [ ]:
def evaluate_stub(question: Question, answer: str) -> dict:
    """Stand-in for interview.evaluate_answer (LLM or heuristic fallback)."""
    score = 70 if len(answer) > 40 else 45
    return {
        "score": score,
        "what_interviewer_wanted": question.intent or "a specific, owned example",
        "strengths": ["Clear ownership language"] if score >= 60 else [],
        "improvements": ["Add metrics"] if score < 80 else [],
    }

def submit(session: Session, answer: str):
    index = len(session.evaluations)
    q = session.questions[index]
    evaluation = evaluate_stub(q, answer)
    session.evaluations.append(evaluation)
    session.turns.append({"role": "candidate", "q": index, "text": answer})
    print(f"Q{index} score={evaluation['score']}")
    next_i = index + 1
    if next_i < len(session.questions):
        ask_question(session, next_i)
    else:
        session.overall_score = round(
            sum(e["score"] for e in session.evaluations) / len(session.evaluations)
        )
        session.completed = True
        print(f"DONE overall={session.overall_score}")

submit(session, "I designed the checkout API, owned deploys, and reduced p95 latency 40%.")
submit(session, "I ran the incident bridge, wrote the RCA, and added a runbook.")
submit(session, "I introduced mypy gradually and gated CI on typed modules.")
print("completed:", session.completed, "score:", session.overall_score)
print("transcript lines:", len(session.turns))


## Persistence

| Store | What |
|-------|------|
| Gradio `iv_state` | `session.model_dump_json()` for the live round-trip |
| SQLite `interview_sessions` | session blob, overall score, completed |
| SQLite `interview_turns` | role + text per turn |

Audio paths may be referenced temporarily; **durable content is text**.


## Code reference

| Concern | File | Symbol |
|---------|------|--------|
| Start | `app.py` | `interview_start` |
| Ask | `app.py` | `_ask_question` |
| Submit | `app.py` | `interview_submit` |
| Questions / eval | `interview.py` | `generate_questions`, `evaluate_answer`, `overall_evaluation` |
| Turns | `interview.py` | `append_turn` |
| Guardrails | `guardrails.py` | `validate_interview_answer`, `validate_evaluation` |

See `01_voice_and_transcript.ipynb` for STT/TTS detail.
